# ResNet-18 Robust

This notebook trains a pretrained ResNet-18 on a robust dataset which combrises of a mix of SID and CIFAKE.

- Label `0`: real
- Label `1`: AI-generated/fake
- Input size: 224 x 224
- Model selection: highest validation F1

## Check that GPU exists

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available")

CUDA available: True
GPU: Tesla T4


## Set up project

In [43]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_ROOT = Path("/content/ai-image-detector")

if not PROJECT_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/mikkichan22/AI-image-detector.git",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

(PROJECT_ROOT / "src" / "__init__.py").touch()

print("Working directory:", Path.cwd())
print("Project exists:", PROJECT_ROOT.exists())
print("src exists:", (PROJECT_ROOT / "src").exists())

Working directory: /content/ai-image-detector
Project exists: True
src exists: True


In [5]:
!pip install -q datasets

## Create sid subset

In [11]:
!python -m src.create_sid_subset

Loading SID_Set in streaming mode...
README.md: 100% 3.30k/3.30k [00:00<00:00, 7.76MB/s]
Resolving data files: 100% 249/249 [00:00<00:00, 447081.21it/s]
Resolving data files: 100% 34/34 [00:00<00:00, 78701.07it/s]
Resolving data files: 100% 249/249 [00:00<00:00, 417085.34it/s]
Resolving data files: 100% 34/34 [00:00<00:00, 264085.81it/s]
Saved: {1: 504, 0: 496}
Saved: {1: 995, 0: 1005}
Saved: {1: 1487, 0: 1513}
Saved: {1: 1967, 0: 2033}
Saved: {1: 2472, 0: 2528}
Saved: {1: 2982, 0: 3018}
Saved: {1: 3494, 0: 3506}
Saved: {1: 4007, 0: 3993}
Saved: {1: 4526, 0: 4474}
Saved: {1: 5029, 0: 4971}
Saved: {1: 5543, 0: 5457}
Saved: {1: 6033, 0: 5967}
Saved: {1: 6537, 0: 6463}
Saved: {1: 7033, 0: 6967}
Saved: {1: 7556, 0: 7444}
Saved: {1: 8019, 0: 7981}
Saved: {1: 8513, 0: 8487}
Saved: {1: 9009, 0: 8991}
Saved: {1: 9527, 0: 9473}
Saved: {1: 10000, 0: 10000}

Finished.
Images saved: {1: 10000, 0: 10000}
Output: /content/ai-image-detector/data/raw/SID_Set_subset


## Check that it exists

In [12]:
!find data/raw/SID_Set_subset -type f | wc -l
!du -sh data/raw/SID_Set_subset

20000
4.5G	data/raw/SID_Set_subset


## Create SID splits

In [41]:
!python -m src.create_sid_splits

Images found: 20000
Duplicate groups: 2
Created: /content/ai-image-detector/data/sid_splits.csv
train      label=0: 8500
train      label=1: 8501
validation label=0: 1500
validation label=1: 1499


In [42]:
!find src -maxdepth 1 -type f | sort

src/create_robustness_sets.py
src/create_sid_splits.py
src/create_sid_subset.py
src/create_splits.py
src/data.py
src/evaluate.py
src/__init__.py
src/inspect_dataset.py
src/train_resnet.py
src/transforms.py


## Create combined splits (SID and CIFAKE)

In [ ]:
!python -m src.combine_splits

## inspect result

In [ ]:
import pandas as pd

combined = pd.read_csv("data/combined_splits.csv")

display(
    combined.groupby(
        ["source_dataset", "split", "label"]
    ).size()
)